# LP and LCS: input polynomials, shared coordinates, and finite checks

**Question.** Which polynomial inputs are being lifted, why does a formal adjoint matter, and which finite parameters are actually certified?

**Paper locations.** Section VI, Definition 4; Appendix D, “Order of the lifted product and binary expansion”; Appendix G.1, “Finite cyclic LP inputs”; and Appendix G.2, “LCS block expansion and parameters” (`def:lp`, `app:lp-order`, `app:lp-examples`, `app:lcs-details`).

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

The paper numbers refer to the manuscript identified by `PAPER_SOURCE` in
[`paper_cases.py`](../qldpc_generator/paper_cases.py). Source labels are retained
to help locate the same material after renumbering. Equation numbers below are
local to this notebook; code outputs are computational results.

**Before running.** Follow the [installation instructions](../README.md#installation-and-execution),
select that Python environment, and run the cells in order. All matrix arithmetic
and ranks are over GF(2); matrix indices start at zero. For the coverage overview,
start with [06 — Paper validation](06-paper-validation.ipynb).

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Fix the ring and the shift direction

The lift size $L$ specifies one shared cyclic coordinate. Each polynomial entry
is expanded into an $L\times L$ binary block by substituting a cyclic shift:

$$
R_L=\mathbb F_2[x]/(x^L-1),\qquad
\mathbb B_L(p)=p(S),\qquad S e_j=e_{j+1\bmod L}. \tag{1}
$$

The next small matrix makes the direction unambiguous. Polynomial exponents
are reduced modulo L, and repeated terms cancel modulo two.

In [2]:
import sympy as sp
from qldpc_generator.algebra import (
    x, polynomial_to_circulant, formal_adjoint, cyclic_coefficients,
    cyclic_group, expand_group_matrix,
)
from qldpc_generator import lifted_product, lcs, hgp

S = polynomial_to_circulant(x, 3)
print("S for L=3:\n", S)
assert S[:, 0].tolist() == [0, 1, 0]
assert np.array_equal(polynomial_to_circulant(x**3, 3), np.eye(3, dtype=np.uint8))
assert np.array_equal(polynomial_to_circulant(1+x+x**4, 3), np.eye(3, dtype=np.uint8))

S for L=3:
 [[0 0 1]
 [1 0 0]
 [0 1 0]]


## 2. Transpose the matrix and invert the shifts

Transposing a binary circulant reverses its shift. The formal adjoint must
therefore include both operations:

$$
M^*=\overline M^{\mathsf T},\qquad \overline x=x^{-1},\qquad
\mathbb B_L(M^*)=\mathbb B_L(M)^{\mathsf T}. \tag{2}
$$

The failure control uses ordinary polynomial transposition without the
involution. It has the correct shape but the wrong entries.

In [3]:
M = sp.Matrix([[1+x, x**2, 0], [1, 0, 1+x**3]])
group = cyclic_group(5)
def expand(matrix):
    return expand_group_matrix(group, cyclic_coefficients(matrix, 5))

expected = expand(M).T
correct = expand(formal_adjoint(M))
wrong = expand(M.T)
assert np.array_equal(correct, expected)
assert not np.array_equal(wrong, expected)
show_table(["Operation before expansion", "Shape", "Mismatch count against binary transpose"], [
    ["Formal adjoint", correct.shape, np.count_nonzero(correct != expected)],
    ["Transpose without shift inversion", wrong.shape, np.count_nonzero(wrong != expected)],
])

| Operation before expansion | Shape | Mismatch count against binary transpose |
| --- | --- | --- |
| Formal adjoint | (15, 10) | 0 |
| Transpose without shift inversion | (15, 10) | 30 |

## 3. A shared lift is not two independent lifts

For polynomial inputs of sizes $r_A\times n_A$ and $r_B\times n_B$, LP retains
one group coordinate. An ordinary HGP of already expanded matrices retains two:

$$
N_{\rm LP}=(n_A r_B+r_A n_B)L,\qquad
N_{\rm independent}=(n_A r_B+r_A n_B)L^2. \tag{3}
$$

Use the one-entry inputs A = 1+x and B = 1+x² at L = 3. The second HGP input
is transposed to match the chain orientation of B. This comparison concerns
construction size; it does not assert that the two codes are equivalent.

In [4]:
shared = lifted_product([[1+x]], [[1+x**2]], 3)
independent = hgp(polynomial_to_circulant(1+x, 3),
                  polynomial_to_circulant(1+x**2, 3).T)
assert (shared.n, independent.n) == (6, 18)
show_table(["Construction", "Cyclic coordinates", "N", "H_X shape"], [
    ["LP over R_3, then expansion", 1, shared.n, shared.hx.shape],
    ["Expand both factors, then ordinary HGP", 2, independent.n, independent.hx.shape],
])

| Construction | Cyclic coordinates | N | H_X shape |
| --- | --- | --- | --- |
| LP over R_3, then expansion | 1 | 6 | (3, 6) |
| Expand both factors, then ordinary HGP | 2 | 18 | (9, 18) |

## 4. Reproduce the three published cyclic LP inputs

Appendix G.1 gives three 3-by-5 matrices. The arrays below transcribe their
exponents: an entry 0 denotes the monomial 1, not a zero polynomial.
For each lift, A = M and B = M*; there are 34L qubits and 15L rows in each
quantum check matrix. The polynomial matrices are displayed before construction
so that the reader can compare every input with the paper.

The library's stored cases are checked against these explicit inputs. The
reported distance upper bounds are shown separately; this calculation does not
produce logical witnesses for those bounds.

In [5]:
from qldpc_generator.paper_cases import cyclic_lp_case
exponents = {
    16: [[0,0,0,0,0], [0,2,4,7,11], [0,3,10,14,15]],
    21: [[0,0,0,0,0], [0,4,5,7,17], [0,14,18,12,11]],
    30: [[0,0,0,0,0], [0,2,14,24,25], [0,16,11,14,13]],
}
expected = {16: (80, 232, 12), 21: (100, 307, 16), 30: (136, 442, 20)}
rows = []
for L, entries in exponents.items():
    M = sp.Matrix([[x**power for power in row] for row in entries])
    print(f"M_{L} =")
    display(M)
    lp = lifted_product(M, formal_adjoint(M), L)
    stored = cyclic_lp_case(L)
    assert np.array_equal(lp.hx, stored.hx) and np.array_equal(lp.hz, stored.hz)
    K, binary_rank, reported_upper = expected[L]
    p = lp.parameters()
    assert (lp.n, lp.k, p['rank_x'], p['rank_z']) == (34*L, K, binary_rank, binary_rank)
    rows.append([L, lp.hx.shape, lp.n, lp.k, binary_rank, f"D <= {reported_upper}", "not searched"])
show_table(["L", "H_X / H_Z shape", "N", "K", "Each binary rank", "Literature claim", "Local distance"], rows)

M_16 =


Matrix([
[1,    1,     1,     1,     1],
[1, x**2,  x**4,  x**7, x**11],
[1, x**3, x**10, x**14, x**15]])

M_21 =


Matrix([
[1,     1,     1,     1,     1],
[1,  x**4,  x**5,  x**7, x**17],
[1, x**14, x**18, x**12, x**11]])

M_30 =


Matrix([
[1,     1,     1,     1,     1],
[1,  x**2, x**14, x**24, x**25],
[1, x**16, x**11, x**14, x**13]])

| L | H_X / H_Z shape | N | K | Each binary rank | Literature claim | Local distance |
| --- | --- | --- | --- | --- | --- | --- |
| 16 | (240, 544) | 544 | 80 | 232 | D <= 12 | not searched |
| 21 | (315, 714) | 714 | 100 | 307 | D <= 16 | not searched |
| 30 | (450, 1020) | 1020 | 136 | 442 | D <= 20 | not searched |

## 5. Build one LCS instance from its seed

LCS uses a different input family. Here $\ell$ is the number of rows of the
repetition seed, while $L$ is the cyclic lift size. For $\ell=2$ the polynomial
seed is

$$
\widetilde H^{(2)}=
\begin{pmatrix}1&1+x&0\\0&1&1+x\end{pmatrix},\qquad
A=(\widetilde H^{(2)})^*,\quad B=\widetilde H^{(2)}. \tag{4}
$$

The `lcs(ell, L)` constructor uses these inputs and puts the larger qubit block
first. The column permutation below makes that convention explicit.
For this family the paper gives

$$
N=((\ell+1)^2+\ell^2)L,\quad K=L,\qquad
D\stackrel{\text{conjectured}}{=}\min(L,2\ell+1). \tag{5}
$$

At (ell, L) = (2,4), we check the exact finite distance separately from the
general conjecture. The matrices have 24 rows and 52 columns.

In [6]:
ell, L = 2, 4
seed = sp.Matrix([[1, 1+x, 0], [0, 1, 1+x]])
raw_lp = lifted_product(formal_adjoint(seed), seed, L)
cut = ell**2 * L
column_order = np.r_[np.arange(cut, raw_lp.n), np.arange(cut)]
lcs_code = lcs(ell, L)
assert np.array_equal(lcs_code.hx, raw_lp.hx[:, column_order])
assert np.array_equal(lcs_code.hz, raw_lp.hz[:, column_order])
assert lcs_code.hx.shape == lcs_code.hz.shape == (24, 52)
assert (lcs_code.n, lcs_code.k) == (52, 4)
distance = lcs_code.distance(4)
assert distance['status'] == 'exact' and distance['value'] == 4
show_table(["Instance", "N", "K", "Local distance", "General family claim"], [
    ["ell=2, L=4", lcs_code.n, lcs_code.k, "D = 4 (exact)",
     lcs_code.metadata['distance_claim']['status']],
])
show_table(["Sector", "Weight", "Logical support"], [
    [sector, data['value'], data['witness']] for sector, data in distance['sectors'].items()
])

| Instance | N | K | Local distance | General family claim |
| --- | --- | --- | --- | --- |
| ell=2, L=4 | 52 | 4 | D = 4 (exact) | conjectured |

| Sector | Weight | Logical support |
| --- | --- | --- |
| X | 4 | [32, 33, 34, 35] |
| Z | 4 | [32, 33, 34, 35] |

## 6. What this establishes

We checked the lift convention, demonstrated why a plain transpose fails,
reproduced the three large LP matrix ranks, and certified [[52,4,4]] for one
LCS input. The three LP distance upper bounds remain literature claims here;
the LCS family distance expression remains a conjecture.

Implementation: [polynomial algebra](../qldpc_generator/algebra.py),
[constructors](../qldpc_generator/constructions.py), and
[paper-input tests](../tests/test_paper_cases.py).
Continue with [03 — BP orbit bases](03-example-bp.ipynb), then
[04 — Construction correspondences](04-fb-and-correspondences.ipynb).